# Load required packages

To install the packages required for this notebook on the HPC, please follow the 'Jupyter Kernel Creation' slides posted on OPAL.

In [4]:
import re

import pandas as pd
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

# Load the model (Llama-8B or Mistral-7B)

Note that you need to be on the partition with GPU (e.g. capella, alpha).

In [5]:
device = "cuda"

This is the model which doesn't require requesting access. If you have the access to the Llama-8B model, you can use it instead.

In [6]:
model_name = "meta-llama/Meta-Llama-3-8B-Instruct"
HF_TOKEN = "<HF_TOKEN_REDACTED>"

In [7]:
from huggingface_hub import login
login(token=HF_TOKEN)

tokenizer = AutoTokenizer.from_pretrained(model_name, token=HF_TOKEN)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


In [9]:
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.bfloat16,
    token=HF_TOKEN,
).to(device)

Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

RuntimeError: Found no NVIDIA driver on your system. Please check that you have an NVIDIA GPU and installed a driver from http://www.nvidia.com/Download/index.aspx

In [ ]:
DATA_DIR = "/data/cat/ws/jipe574g-jie/homework_llm/project/data"
OUTPUT_DIR = "/data/cat/ws/jipe574g-jie/homework_llm/project/output"

import os
os.makedirs(DATA_DIR, exist_ok=True)
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Load training data for reference
train_mcq = pd.read_csv(f"{DATA_DIR}/train_dataset_mcq.csv")
train_saq = pd.read_csv(f"{DATA_DIR}/train_dataset_saq.csv")

print(f"Training MCQ: {len(train_mcq)} questions")
print(f"Training SAQ: {len(train_saq)} questions")

# SAQ Task

In [ ]:
# Select few-shot examples from training data, categorized by country
def get_diverse_examples(train_df, n_per_country=4, task_type="saq"):
    """Select diverse training samples covering different countries"""
    examples = ""

    # Get column names for question and answer
    if task_type == "saq":
        q_col = 'en_question' if 'en_question' in train_df.columns else 'question'
        a_col = 'en_answer' if 'en_answer' in train_df.columns else 'answer'
    else:  # mcq
        q_col = 'prompt' if 'prompt' in train_df.columns else 'question'
        a_col = 'answer' if 'answer' in train_df.columns else 'correct_answer'

    # Try to select by grouping by country
    if 'country' in train_df.columns:
        priority_countries = ['Iran', 'China', 'Japan', 'India', 'USA', 'Brazil', 'Nigeria']
        for country in priority_countries:
            country_df = train_df[train_df['country'].str.contains(country, case=False, na=False)]
            if len(country_df) >= n_per_country:
                samples = country_df.sample(n=n_per_country, random_state=42)
                for _, row in samples.iterrows():
                    examples += f"Question: {row[q_col]}\nAnswer: {row[a_col]}\n\n"
    else:
        samples = train_df.sample(n=min(20, len(train_df)), random_state=42)
        for _, row in samples.iterrows():
            examples += f"Question: {row[q_col]}\nAnswer: {row[a_col]}\n\n"
    return examples

few_shot_examples_saq = get_diverse_examples(train_saq, n_per_country=4, task_type="saq")


def saq_func(query: str):
    system_prompt = f"""You are a cultural expert. For each question, answer with ONLY the answer word or phrase (2-4 words). Do NOT write 'assistant', 'Answer:', or any explanation.\n\nHere are some examples:\n{few_shot_examples_saq}Now answer the following question in the same way.\n"""
    user_prompt = f"Question: {query}\nAnswer:"
    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt}
    ]
    prompt = tokenizer.apply_chat_template(messages, tokenize=False)
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=20,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
        )
    generated = tokenizer.decode(
        outputs[0][inputs["input_ids"].shape[-1]:],
        skip_special_tokens=True
    ).strip()
    print(f"Q: {query[:80]}...")
    print(f"Raw output: {generated}")
    print("-"*10)

    # Extract all meaningful words after 'assistant'
    answer = generated.strip()
    # Find content after 'assistant'
    match = re.search(r"assistant[\s:：-]*(.*)", answer, re.IGNORECASE)
    if match:
        answer = match.group(1).strip()
    # Remove prefixes like 'answer:'
    for bad in ["answer:", "answer", ":", "-", "="]:
        if answer.lower().startswith(bad):
            answer = answer[len(bad):].strip()
    # Take only the first line and first sentence
    answer = answer.split('\n')[0].strip()
    answer = re.split(r'[.!?]', answer)[0].strip()
    # Remove quotes and colons
    answer = answer.strip('"\'：:')
    # If still incorrect, keep the first 2-4 valid words
    words = [w for w in answer.split() if w.lower() not in ["assistant", "answer", ":", "-", "="]]
    if len(words) == 0:
        answer = "unknown"
    elif len(words) == 1:
        answer = words[0]
    else:
        answer = " ".join(words[:4])
    return answer.lower().strip()

In [ ]:
saq = pd.read_csv(f"{DATA_DIR}/test_dataset_saq.csv")
saq = saq[["ID", "en_question"]]
print(f"Total SAQ questions: {len(saq)}")

In [ ]:
preds = []
for q in saq["en_question"]:
    answer = saq_func(q)
    preds.append(answer)

saq["answer"] = preds

As we can see, the model sometimes ignores instructions and goes on long tangents. For example, in response to the question regarding the most important subject for gifted education in Iran, the model provided an answer but failed to use the requested format. The extraction of the answer is not trivial and left out of scope.

In [ ]:
saq.head(10)

In [ ]:
saq_submission = saq[["ID", "answer"]]
saq_submission.to_csv(f"{OUTPUT_DIR}/saq_prediction.tsv", sep='\t', index=False)
print(f"SAQ saved to: {OUTPUT_DIR}/saq_prediction.tsv")

# MCQ Task

In [ ]:
# Get few-shot examples for MCQ (4 per country)
few_shot_examples_mcq = get_diverse_examples(train_mcq, n_per_country=4, task_type="mcq")
print(f"MCQ few-shot examples length: {len(few_shot_examples_mcq)} chars")

def mcq_func(query: str):
    system_prompt = f"""You are an expert in cultural knowledge and traditions from around the world, including Iran, China, Japan, India, and other countries.

Select ONLY ONE from the given alphabet choices (A, B, C, or D).
Respond with just the letter.

Here are some examples:
{few_shot_examples_mcq}
Follow the same format. Give only the letter (A, B, C, or D), nothing else."""

    user_prompt = f"""Question: {query}

Answer (just the letter):"""

    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt}
    ]

    prompt = tokenizer.apply_chat_template(messages, tokenize=False)
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=20,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
        )

    generated = tokenizer.decode(
        outputs[0][inputs["input_ids"].shape[-1]:],
        skip_special_tokens=True
    ).strip()

    print(f"Q: {query[:80]}...")
    print(f"A: {generated}")
    print("-"*10)

    return generated

In [ ]:
mcq = pd.read_csv(f"{DATA_DIR}/test_dataset_mcq.csv")
mcq = mcq[["MCQID", "prompt"]]
print(f"Total MCQ questions: {len(mcq)}")

In [ ]:
preds = []
for q in mcq["prompt"]:
    answer = mcq_func(q)
    preds.append(answer)

mcq["answer"] = preds
mcq.head(10)

Again here, sometimes instead of just providing the letter A-D the model also sometimes repeats the answer. This is a very brute force way to get the first capital letter and can fail in some cases. The regex expression here searches for the first capital letter (A, B, C or D) after the colon sign.

In [ ]:
mcq["choice"] = mcq["answer"].apply(lambda x: ''.join(re.findall(r'[A-D]', x))[0] if re.findall(r'[A-D]', x) else 'A')

All choices through A to D need to be picked at least ones for this code to create correct dataframe.

In [ ]:
mcq_submission = pd.get_dummies(mcq["choice"]).astype(bool)
mcq_submission = pd.concat([mcq["MCQID"], mcq_submission], axis=1)

In [ ]:
mcq_submission.head()

In [ ]:
mcq_submission.to_csv(f"{OUTPUT_DIR}/mcq_prediction.tsv", sep='\t', index=False)
print(f"MCQ saved to: {OUTPUT_DIR}/mcq_prediction.tsv")